# RAG Evaluation (Deep Dive)
### Practice Notebook

**Assumed pre-installed libraries:** `numpy`, `scikit-learn`,
`sentence-transformers`, `deepeval`

This notebook extends Week 2, Day 5's faithfulness/groundedness heuristics
into the full four-metric RAGAS-style framework: faithfulness, answer
relevance, context precision, and context recall.


## 1. The 2x2 structure of RAG evaluation

| | Evaluates | Metric |
|---|---|---|
| **Generation-side** | Did the model use the context well? | Faithfulness |
| **Generation-side** | Did the model answer the actual question? | Answer relevance |
| **Retrieval-side** | Is the retrieved context clean (not noisy)? | Context precision |
| **Retrieval-side** | Is the retrieved context complete (nothing missing)? | Context recall |

The value of this decomposition: a low overall score tells you a RAG
pipeline has a problem; knowing *which* of these four is low tells you
*where* to fix it. Low context recall -> fix chunking/retrieval. Low
faithfulness -> fix the generation prompt or model. Let's build all four
from scratch, using a small worked example pipeline.


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

from deepeval import evaluate
from deepeval.test_case import LLMTestCase

from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualRecallMetric,
    ContextualPrecisionMetric,
    ContextualRelevancyMetric,
    ToxicityMetric,
)

In [ ]:
judge_llm = # Todo: Get model1(reasoning model) from Hugging Face
judge_llm1 = # Todo: Get model2(reasoning model) from Hugging Face
metrics = [
    FaithfulnessMetric(model=judge_llm),
    AnswerRelevancyMetric(model=judge_llm, include_reason=False),
    ContextualRecallMetric(model=judge_llm),
    ContextualPrecisionMetric(model=judge_llm1),
    ContextualRelevancyMetric(model=judge_llm1)
]

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

# A worked RAG example: query, retrieved chunks, and a generated answer.
query = "What is our policy on remote work for new employees?"

retrieved_chunks = [
    "New employees must complete a 90-day onboarding period in the office "
    "before being eligible for remote work arrangements.",
    "Our office is located in downtown Bengaluru with free parking available.",  # noise
    "Remote work requests are approved by the employee's direct manager.",
]

# A ground-truth reference of what SHOULD be in the retrieved context to
# fully answer the query (used only for context recall -- in a real system
# this often comes from a human-labeled test set).
ground_truth_facts = [
    "New employees must complete a 90-day onboarding period in the office "
    "before being eligible for remote work.",
    "Remote work requests are approved by the employee's direct manager.",
    "Remote work is limited to 2 days per week after the onboarding period.",  # MISSING from retrieval
]

generated_answer = (
    "New employees need to complete 90 days in the office before they can "
    "work remotely. After that, remote work requests go through your manager."
)


## 2. Context precision: is the retrieved context clean?

Of the chunks that were retrieved, what proportion are actually relevant to
the query? We approximate "relevant" here with a similarity threshold
against the query(recall Week 2's discussion of embedding anisotropy).


In [ ]:
def context_precision(query: str, retrieved_chunks: list, threshold: float = 0.6) -> dict:
    """NOTE: 0.6 is a starting point, not a universally safe number -- see
    Exercise 2.1. Unrelated sentence pairs commonly score 0.4-0.6 cosine
    similarity with MiniLM-style embeddings (an effect called anisotropy,
    covered in Week 2), so any fixed threshold needs to be checked against
    real relevant/irrelevant examples for your actual data before trusting it.
    """
    query_emb = model.encode([query])
    chunk_embs = model.encode(retrieved_chunks)
    sims = cosine_similarity(query_emb, chunk_embs)[0]

    relevant_flags = [float(s) >= threshold for s in sims]
    precision = sum(relevant_flags) / len(retrieved_chunks)

    details = [{"chunk": c[:60] + "...", "similarity": round(float(s), 3), "relevant": r}
               for c, s, r in zip(retrieved_chunks, sims, relevant_flags)]
    return {"context_precision": precision, "details": details}

precision_result = context_precision(query, retrieved_chunks)
print("Context precision:", round(precision_result["context_precision"], 3))
for d in precision_result["details"]:
    print(f"  [{'RELEVANT' if d['relevant'] else 'NOISE'}] sim={d['similarity']}  {d['chunk']}")


In [ ]:
## Deepeval metrics 
m = metrics[3]
contextual_precision_testcase = LLMTestCase(
    input=query,
    actual_output = generated_answer,
    expected_output="New employees must complete a 90-day in-office onboarding period before becoming eligible to work remotely up to two days per week. All remote work arrangements require prior approval from the employee's direct manager.",
    retrieval_context=retrieved_chunks
)

evaluate([contextual_precision_testcase], m)


**Exercise 2.1:** Look at the raw `similarity` scores printed above, not
just the `relevant`/`NOISE` labels. Is the parking/office-location chunk's
similarity score meaningfully lower than the other two chunks, or is it
closer than you'd expect (recall Week 2's anisotropy discussion -- unrelated
text often still scores 0.4-0.6)? Try `threshold` values of `0.4`, `0.6`,
and `0.8`:
1. At which threshold (if any) does the parking chunk get correctly flagged
   as noise while the two genuinely relevant chunks are still kept?
2. Now try `threshold = 0.0` -- what happens to `context_precision`, and
   why is a precision score of 1.0 (achieved by calling everything
   "relevant") not actually useful information?
3. Based on what you found, is embedding similarity alone a reliable way to
   judge relevance here, or would you want a second check (e.g., an
   LLM-as-judge call) before trusting this metric in a real system? This
   mirrors the abstention-threshold problem from Week 2, Day 5: a threshold
   that's too permissive makes the metric meaningless, and the "right"
   threshold isn't something you can pick without checking it against real
   examples first.


## 3. Context recall: is the retrieved context complete?

Of everything actually needed to answer the question (the ground truth
facts), how much made it into the retrieved context? This requires a
labeled reference -- in this toy example, `ground_truth_facts` -- which is
exactly why building good test sets (Day 1's rubric-building skill, applied
to retrieval) matters for real RAG evaluation.


In [ ]:
def context_recall(ground_truth_facts: list, retrieved_chunks: list, threshold: float = 0.6) -> dict:
    # NOTE: as with context_precision, calibrate this threshold against
    # real examples rather than trusting the default -- see Exercise 2.1.
    fact_embs = model.encode(ground_truth_facts)
    chunk_embs = model.encode(retrieved_chunks)
    sims = cosine_similarity(fact_embs, chunk_embs)

    covered_flags = [float(np.max(row)) >= threshold for row in sims]
    recall = sum(covered_flags) / len(ground_truth_facts)

    details = [{"fact": f[:60] + "...", "best_match_score": round(float(np.max(row)), 3),
                "covered": c}
               for f, row, c in zip(ground_truth_facts, sims, covered_flags)]
    return {"context_recall": recall, "details": details}

recall_result = context_recall(ground_truth_facts, retrieved_chunks)
print("Context recall:", round(recall_result["context_recall"], 3))
for d in recall_result["details"]:
    print(f"  [{'COVERED' if d['covered'] else 'MISSING'}] score={d['best_match_score']}  {d['fact']}")


Notice the "2 days per week" limit is correctly flagged as **missing** --
it was never retrieved, so no matter how good the generator is, it cannot
mention this fact. This is a retrieval failure, not a generation failure.

**Exercise 2.2:** Add the missing fact to `retrieved_chunks` as a fourth
chunk, re-run `context_recall`, and confirm it now scores 1.0. Then
re-run `context_precision` on the updated `retrieved_chunks` too -- did
adding a genuinely relevant chunk change the precision score? Should it?


## 4. Faithfulness: did the generation stay grounded?

The fraction of claims in the generated answer that can actually be
inferred from the retrieved context (not the ground truth -- the context
the generator actually had access to).


In [ ]:
def faithfulness(answer: str, retrieved_chunks: list, threshold: float = 0.6) -> dict:
    # NOTE: as with context_precision, calibrate this threshold against
    # real examples rather than trusting the default -- see Exercise 2.1.
    claims = [c.strip() for c in answer.split(".") if c.strip()]
    claim_embs = model.encode(claims)
    chunk_embs = model.encode(retrieved_chunks)
    sims = cosine_similarity(claim_embs, chunk_embs)

    supported_flags = [float(np.max(row)) >= threshold for row in sims]
    score = sum(supported_flags) / len(claims)

    details = [{"claim": c, "best_match_score": round(float(np.max(row)), 3), "supported": s}
               for c, row, s in zip(claims, sims, supported_flags)]
    return {"faithfulness": score, "details": details}

faith_result = faithfulness(generated_answer, retrieved_chunks)
print("Faithfulness:", round(faith_result["faithfulness"], 3))
for d in faith_result["details"]:
    print(f"  [{'SUPPORTED' if d['supported'] else 'UNSUPPORTED'}] score={d['best_match_score']}  {d['claim']}")


## 5. Answer relevance: did the answer address the actual question?

Measured as semantic similarity between the generated answer and the
original query. Note this is a different comparison than faithfulness
(answer vs. context) -- an answer can be highly faithful to its context
while still failing to actually address what was asked (recall Day 1's
"correct but unhelpful" example).


In [ ]:
def answer_relevance(query: str, answer: str) -> float:
    query_emb = model.encode([query])
    answer_emb = model.encode([answer])
    return float(cosine_similarity(query_emb, answer_emb)[0][0])

relevance_score = answer_relevance(query, generated_answer)
print("Answer relevance:", round(relevance_score, 3))


## 6. Putting it together: a full RAG evaluation report

In [ ]:
def evaluate_rag_pipeline(query, retrieved_chunks, generated_answer, ground_truth_facts):
    # Todo: Execute the metrices with Deepeval framework
    return {
        "context_precision": context_precision(query, retrieved_chunks)["context_precision"],
        "context_recall": context_recall(ground_truth_facts, retrieved_chunks)["context_recall"],
        "faithfulness": faithfulness(generated_answer, retrieved_chunks)["faithfulness"],
        "answer_relevance": answer_relevance(query, generated_answer),
    }

report = evaluate_rag_pipeline(query, retrieved_chunks, generated_answer, ground_truth_facts)
print("=== RAG Evaluation Report ===")
for metric, score in report.items():
    print(f"  {metric:20s}: {round(score, 3)}")

overall = sum(report.values()) / len(report)
print(f"\n  {'ragas_overall (mean)':20s}: {round(overall, 3)}")


**Exercise 2.3 (mini deliverable):** Build a small evaluation with Deepeval framework and put the results in rubric table
(you can do this as a markdown cell) with columns: metric name, what it
measures, which pipeline stage it diagnoses (retrieval or generation), and
a pass/fail threshold you'd propose for a real system. Then run
`evaluate_rag_pipeline` on 2 more query/context/answer examples of your own
-- include at least one example where you deliberately make the retrieval
bad (irrelevant chunks) and one where retrieval is good but the generated
answer contradicts the context. Confirm the four metrics correctly
localize each problem to the stage you intended.


## 7. A note on real automated RAG evaluation frameworks

Compare the results with Deepeval framework and simply by executing semantic similarity. Write the difference.
